In [3]:
import os
import sys
from pytest import skip
import wandb.plot
from xgboost import XGBClassifier
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, precision_recall_curve, balanced_accuracy_score, average_precision_score
from sklearn.metrics import auc as auc_sklearn
import itertools 
import secrets
from sklearn.model_selection import train_test_split
import warnings
import cupy as cp
from cupy.cuda import Device
import random
from joblib import Parallel, delayed
from scipy.optimize import minimize
from scipy.special import expit, logit            # σ(x) = 1 / (1+e^{-x})
from sklearn.isotonic import IsotonicRegression

# Import functions from emc_ensemble_retrain.py
sys.path.append('/users/gzanardini/eeg_thesis/emc/')
from emc_ensemble_retrain import (
    train_simplex_logistic, SimplexLogistic, setup_environment, 
    find_optimal_threshold, calculate_bac,
    handle_complex_numbers, load_data, load_feature_data,
    get_train_val_test_indices, preload_all_feature_data,
    get_cached_feature_data, train_feature_model_parallel,
    train_feature_model, train_ensemble_models, save_results
)

np.set_printoptions(linewidth=200, precision=4)
warnings.simplefilter(action='ignore', category=FutureWarning)

# Configuration
N_RUNS = 5
N_CUDA = 0
DEVICE = 'cuda'
SPLIT_RATIO = 0.3
PROJECT_NAME = 'emc_size10'
DATA_FOLDER = '/space/gzanardini/emc_whole/split'
LOG_FOLDER = '/space/gzanardini/emc/'
N_JOBS_XGB = 4  # Set to 1 for compatibility with CUDA
NUM_WORKERS = 10  # Number of max parallel workers for training
N_PARALLEL_FEATURES = NUM_WORKERS  # Parallel feature training within combination
SCIPY_ARRAY_API=1  # Enable SciPy array API for compatibility with cupy

montages = ['CAR', 'Cz', 'BipolarDB', 'Laplacian']
segment_lengths = [1, 2, 5, 10, 20, 60]
feature_names = ['cc', 'cwt', 'dwt', 'plv', 'mst', 'sst', 'spectral', 'utm', 'gcc', 'gplv']
combiners = ['mean', 'median', 'std', 'skew', 'kurt']

best_parameters = {
    'spectral': ('Cz',          10 ,    'std'),
    'cwt':      ('BipolarDB',   2,      'median'),
    'dwt':      ('Laplacian',   10,     'median'),
    'mst':      ('BipolarDB',   60,     'median'),
    'sst':      ('CAR',         10,     'median'),
    'cc':       ('CAR',         1,      'std'),
    'plv':      ('Laplacian',   60,      'kurt'),
    'gcc':      ('CAR',         60,      'median'),
    'gplv':     ('Laplacian',   2,      'std'),
    'utm':      ('Laplacian',   20,     'std')
}

def generate_feature_combinations():
    """Generate combinations of features from 10 to all features."""
    combinations = []

    # Generate all combinations of 10 to len(feature_names) features
    for i in range(10, len(feature_names) + 1):
        combs = list(itertools.combinations(feature_names, i))
        for comb in combs:
            combinations.append(list(comb))
    
    return combinations

In [4]:

"""Main execution function with data preloading."""
setup_environment()
description, labels, subjects, unique_subjects, subject_labels = load_data()

# Preload all feature data once
preload_all_feature_data()

# Generate all feature combinations once before starting runs
all_combinations = generate_feature_combinations()
print(f"Generated {len(all_combinations)} feature combinations to evaluate")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /users/gzanardini/.netrc


Preloading all feature data...
Loaded cc: (141, 171)
Loaded cwt: (141, 468)
Loaded dwt: (141, 456)
Loaded plv: (141, 1026)
Loaded mst: (141, 108)
Loaded sst: (141, 114)
Loaded spectral: (141, 90)
Loaded utm: (141, 247)
Loaded gcc: (141, 20)
Loaded gplv: (141, 20)
Generated 1 feature combinations to evaluate


In [ ]:

# Evaluate each feature combination
combination_name = '+'.join(combination)
print(f"\nEvaluating combination: {combination_name}")

# Store weights from all runs for this combination
all_run_weights = []

for run_n in range(N_RUNS):

    seed = secrets.randbelow(5000)
    random.seed(seed)
    np.random.seed(seed)
    cp.random.seed(seed)

    RUN_NAME = f'{combination_name}_run_{run_n}'


    print(f'RUN {run_n+1}/{N_RUNS} - Seed: {seed}')
    
    # Initialize arrays to store predictions for this combination
    y_true_all = []
    y_pred_all = []
    y_prob_all = []
    subject_ids = []
    fold_weights = []  # Store weights from each fold
                
    # Iterate through all subjects (LOSO)
    for subject in unique_subjects:
        
        # Leave current subject out for testing
        train_idxs, val_idxs, test_idxs = get_train_val_test_indices(
            description, labels, subject, seed
        )
        
        y_train = labels[train_idxs]
        y_val = labels[val_idxs]
        y_test = labels[test_idxs]
        
        # Train ensemble models for this feature combination (now with parallelization)
        ensemble_result = train_ensemble_models(
            combination, train_idxs, val_idxs, test_idxs, y_train, y_val, seed
        )

        print(f'LR Weights: {ensemble_result["lr_weights"]}')
        
        # Store weights from this fold
        fold_weights.append(ensemble_result["lr_weights"])
        
        # Store predictions for this subject
        y_true_all.extend(y_test)
        y_pred_all.extend(ensemble_result['test_preds'])
        y_prob_all.extend(ensemble_result['test_probs'])
        subject_ids.extend([subject] * len(y_test))
        
    # After LOSO CV is complete for this combination, calculate overall metrics
    print(f"LOSO CV complete for combination: {combination_name}, calculating metrics...")
    
    # Calculate weight statistics across folds for this run
    fold_weights = np.array(fold_weights)  # Shape: (n_folds, n_features)
    mean_weights_run = np.mean(fold_weights, axis=0)
    
    # Store the mean weights from this run
    all_run_weights.append(mean_weights_run)
    
    print(f"Mean weights across folds for run {run_n}: {mean_weights_run}")
    
    # Calculate overall performance
    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_prob_all = np.array(y_prob_all)
    
    auc = roc_auc_score(y_true_all, y_prob_all)
    bac = balanced_accuracy_score(y_true_all, y_pred_all)
    bac80, fpr, tpr, _ = calculate_bac(y_true_all, y_prob_all, 0.8)
    accuracy = accuracy_score(y_true_all, y_pred_all)
    precision = precision_score(y_true_all, y_pred_all)
    recall = recall_score(y_true_all, y_pred_all)
    f1 = f1_score(y_true_all, y_pred_all)
    
    # Log results
    
    roc_data = [[f, t] for f, t in zip(fpr, tpr)]
    roc_table = wandb.Table(data=roc_data, columns=["fpr", "tpr"])
    roc_line = wandb.plot.line(roc_table, "fpr", "tpr", title="ROC Curve")
    
    p, r, t = precision_recall_curve(y_true_all, y_prob_all)
    pr_data = [[f, t] for f, t in zip(r, p)]
    pr_table = wandb.Table(data=pr_data, columns=["precision", "recall"])
    pr_line = wandb.plot.line(pr_table, "precision", "recall", title="Precision-Recall Curve")
    
    auprc = auc_sklearn(r, p)
    ap = average_precision_score(y_true_all, y_prob_all)

    # Prepare results DataFrame
    results = {
        'run': run_n,
        'seed': seed,
        'auc': auc,
        'bac': bac,
        'bac80': bac80,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'auprc': auprc,
        'AP': ap,
    }
    
    # Add weight statistics to results (fold-level stats for this run)
    for i, feature in enumerate(combination):
        results[f'{feature}_mean_weight'] = mean_weights_run[i]
        results[f'{feature}_std_weight'] = np.std(fold_weights[:, i])
    
    results_df = pd.DataFrame([results])
    # Prepare predictions DataFrame
    predictions = {
        'subject': subject_ids,
        'y_true': y_true_all,
        'y_pred': y_pred_all,
        'y_prob': y_prob_all
    }
    predictions_df = pd.DataFrame(predictions)
    # Save results and predictions
    save_results(results_df, predictions_df, RUN_NAME, run_n, seed)
    print(f"Results for combination {combination_name} saved successfully.")
    # Finish wandb run
    print(f"Run {RUN_NAME} completed successfully.")

# After all runs are complete, create plot with run-averaged weights
all_run_weights = np.array(all_run_weights)  # Shape: (n_runs, n_features)
mean_weights_across_runs = np.mean(all_run_weights, axis=0)
std_weights_across_runs = np.std(all_run_weights, axis=0)

print(f"Mean weights across runs: {mean_weights_across_runs}")
print(f"Std weights across runs: {std_weights_across_runs}")

# Create bar plot with error bars for feature weights (averaged across runs)
plt.figure(figsize=(12, 6))
x_pos = np.arange(len(combination))
plt.bar(x_pos, mean_weights_across_runs, yerr=std_weights_across_runs, capsize=5, 
        alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.5)

# Add standard deviation values as text annotations
for i, (mean_w, std_w) in enumerate(zip(mean_weights_across_runs, std_weights_across_runs)):
    plt.text(i, mean_w + std_w + 0.01, f'±{std_w:.3f}', 
            ha='center', va='bottom', fontsize=9, 
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

plt.xlabel('Features')
plt.ylabel('Weight')
plt.title(f'Feature Weights Averaged Across {N_RUNS} Runs\n{combination_name}')
plt.xticks(x_pos, combination, rotation=45, ha='right')
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save the plot
plot_filename = f'{LOG_FOLDER}/{PROJECT_NAME}/{combination_name}_weights_across_runs.png'
os.makedirs(os.path.dirname(plot_filename), exist_ok=True)
plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
plt.close()

print(f"Weights plot for combination {combination_name} saved successfully.")

wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.


wandb: Currently logged in as: giacomo-zanardinibs (EEGthesis). Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /users/gzanardini/.netrc
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /users/gzanardini/.netrc


Preloading all feature data...
Loaded cc: (141, 171)
Loaded cwt: (141, 468)
Loaded dwt: (141, 456)
Loaded plv: (141, 1026)
Loaded mst: (141, 108)
Loaded sst: (141, 114)
Loaded spectral: (141, 90)
Loaded utm: (141, 247)
Loaded gcc: (141, 20)
Loaded gplv: (141, 20)
Generated 1 feature combinations to evaluate

Evaluating combination: cc+cwt+dwt+plv+mst+sst+spectral+utm+gcc+gplv
RUN 1/5 - Seed: 4864
Stage 1: Training individual models and learning meta-learner weights...
Stage 1 - Meta-learner weights: [7.7544e-15 0.0000e+00 1.4561e-01 2.9174e-01 0.0000e+00 2.5812e-01 4.5552e-02 0.0000e+00 1.4780e-01 1.1118e-01]
Stage 1 - Validation AUC: 0.9111, BAC: 0.6667, BAC80: 0.8333
Stage 2: Retraining models on train+val data for final predictions...
Stage 1 - Meta-learner weights: [7.7544e-15 0.0000e+00 1.4561e-01 2.9174e-01 0.0000e+00 2.5812e-01 4.5552e-02 0.0000e+00 1.4780e-01 1.1118e-01]
Stage 1 - Validation AUC: 0.9111, BAC: 0.6667, BAC80: 0.8333
Stage 2: Retraining models on train+val data fo

In [ ]:
# print the weights
# print(f"Learned weights: {w_simplex}")